<a href="https://colab.research.google.com/github/FelipeCJ07/Projeto_Alura_BurnoutCheck-IA-Multimodal/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Projeto: BurnoutCheck IA - Check-in Multimodal Experimental

Este notebook Google Colab implementa uma ferramenta experimental de check-in de bem-estar, explorando a capacidade multimodal (texto e imagem) do modelo Gemini do Google, combinada com técnicas básicas de visão computacional para análise visual preliminar. O objetivo é criar um protótipo interativo que permite ao usuário descrever seus sentimentos e enviar uma selfie para uma avaliação cautelosa e não-diagnóstica baseada na análise dos inputs.

⚠️ AVISO IMPORTANTE E ÉTICA ⚠️

Este é um projeto EXPERIMENTAL, protótipo e educacional, criado exclusivamente para fins de demonstração de capacidades de IA multimodal e visão computacional.

    - A análise fornecida pela Inteligência Artificial (IA) é baseada em padrões identificados nos dados em que o modelo foi treinado e não constitui, em hipótese alguma, um diagnóstico médico, psicológico ou profissional.
    - As sugestões de autocuidado e recursos são genéricas e informativas, não substituem o aconselhamento personalizado de um especialista qualificado.
    - A detecção facial é uma funcionalidade técnica para processamento da imagem e não implica qualquer julgamento ou análise profissional sobre a aparência.
    - A precisão da análise multimodal da IA pode variar e é limitada pela qualidade dos inputs (texto e imagem) e pelas capacidades inerentes ao modelo.

🚫🚫🚫 ESTA FERRAMENTA NÃO FAZ DIAGNÓSTICOS! 🚫🚫🚫

SEMPRE, em caso de preocupações reais com saúde mental, cansaço extremo, estresse crônico ou qualquer outro problema de saúde, PROCURE IMEDIATAMENTE UM PROFISSIONAL DE SAÚDE QUALIFICADO (médico, psicólogo, psiquiatra, etc.). A responsabilidade pelo uso das informações geradas por esta ferramenta é inteiramente do usuário, que deve estar ciente de suas limitações estritas e propósito demonstrativo.

Tecnologias Utilizadas

    - Google Colab: Ambiente de desenvolvimento baseado em nuvem.
    - Python: Linguagem de programação principal.
    - Google Generative AI (google-generativeai): SDK para interação com a API do modelo Gemini (especificamente gemini-1.5-flash-latest para capacidade multimodal).
    - Gradio (gradio): Biblioteca para criação rápida de interfaces web interativas, permitindo a demonstração em um link público a partir do Colab.
    - OpenCV (opencv-python): Biblioteca de Visão Computacional para processamento de imagem e detecção facial.
    - Pillow (PIL): Biblioteca para manipulação de imagens.
    - Outras Bibliotecas Padrão: os, io, json, datetime, time, textwrap, numpy, urllib.request.



Setup e Execução do Notebook

Para rodar este projeto, execute as células do notebook Google Colab na ordem sequencial, de cima para baixo.

Passos Essenciais:

    * Abra este arquivo .ipynb no Google Colab.
    * Configure sua Chave de API do Google Gemini:
        - Obtenha uma chave de API na Google AI Studio (https://aistudio.google.com/app/apikey).
        - No painel lateral esquerdo do Colab, clique no ícone de chave (🔑) para abrir "Secrets" (Segredos).
        - Crie um novo segredo com o nome API_KEY.
        - Cole sua chave de API no campo "Value" (Valor).
        - Certifique-se de que a chave "Notebook access to secrets" (Acesso do notebook aos segredos) esteja ativada para este notebook.
    * Execute cada célula de código na ordem. A saída de cada célula indicará o status da etapa. A interface Gradio será lançada na última célula.

Célula 1 : Instalação de Bibliotecas Essenciais

Esta célula garante que todas as bibliotecas Python necessárias para o projeto (Google Generative AI, Pillow, Gradio e OpenCV) estejam instaladas no ambiente de execução do Colab. A instalação do Gradio é forçada para uma versão mais recente (>=3.14) para compatibilidade com os recursos de layout utilizados.


In [74]:
# Célula 1 : Instalação de Bibliotecas Essenciais

# Instalar bibliotecas necessárias: google-generativeai para Gemini, Pillow para imagem, gradio para a interface web, opencv-python para visão computacional.
print("--- Célula 1: Instalação de Bibliotecas ---")
print("Verificando e instalando dependências (incluindo Gradio >= 3.14 e OpenCV)...")

try:
    import google.generativeai
    import PIL
    import gradio as gr
    import cv2 # Importar OpenCV
    import numpy as np # Necessário para manipulação de imagem com OpenCV
    # Verifica a versão do Gradio se já estiver instalado
    # gradio_version = gr.__version__ # Opcional: verificar versão
    # print(f"Gradio versão {gradio_version} encontrado.") # Opcional: verificar versão

    print("Dependências já instaladas.")
except ImportError:
    print("Instalando google-generativeai, Pillow, gradio>=3.14, opencv-python...")
    # -q para instalação silenciosa
    # Instala uma versão do Gradio maior ou igual a 3.14 e OpenCV
    !pip install -q google-generativeai Pillow gradio>=3.14 opencv-python
    print("Instalação concluída.")
    # Re-importar tudo para garantir que as versões corretas estão carregadas
    import google.generativeai as genai
    import PIL
    from PIL import Image as PILImage
    import gradio as gr
    import io
    import time
    import textwrap
    import json
    from datetime import datetime
    import os
    from google.colab import userdata
    from IPython.display import display, Image, clear_output
    import cv2 # Re-importar OpenCV
    import numpy as np # Re-importar NumPy


print("\nImports e verificação de instalação concluídos.")
print("Célula 1 executada: Imports e verificação de instalação OK.")

--- Célula 1: Instalação de Bibliotecas ---
Verificando e instalando dependências (incluindo Gradio >= 3.14 e OpenCV)...
Dependências já instaladas.

Imports e verificação de instalação concluídos.
Célula 1 executada: Imports e verificação de instalação OK.


Célula 2 : Imports Adicionais, Variáveis Globais e Função de Detecção Facial

Esta célula define variáveis globais que armazenarão a API Key, o modelo Gemini, a sessão de chat, o histórico textual e o caminho do arquivo de histórico. Mais importante, ela define a função detect_and_draw_face, que utiliza a biblioteca OpenCV para realizar a detecção de rostos em uma imagem e desenha um retângulo ao redor deles. Inclui também uma função auxiliar para baixar o arquivo classificador Haar Cascade necessário para a detecção.



In [75]:
# Célula 2 : Imports Adicionais, Variáveis Globais e Função de Detecção Facial

print("--- Célula 2: Imports Adicionais, Variáveis Globais e Função de Detecção Facial ---")

# Imports adicionais
import io
import time
import textwrap
import json
from datetime import datetime
import os
from google.colab import userdata # Necessário para secrets
from PIL import Image as PILImage
import google.generativeai as genai
import cv2 # Importar OpenCV
import numpy as np # Importar NumPy
import urllib.request # Para baixar o arquivo cascade

# --- Variáveis Globais ---
api_key = None
model = None
chat_session = None
chat_history_list = []
HISTORY_FILE = 'burnoutcheck_history.jsonl'

# Caminho para o classificador Haar Cascade para detecção de face frontal
# Este arquivo geralmente vem com a instalação do OpenCV, mas o caminho pode variar.
# Vamos baixá-lo para um local conhecido se não existir.
HAAR_CASCADE_FACE_PATH = '/content/haarcascade_frontalface_default.xml'
HAAR_CASCADE_URL = 'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml'


# --- Função para Baixar o Arquivo Haar Cascade ---
def download_cascade_file(url, filename):
    """Baixa um arquivo se ele não existir localmente."""
    if not os.path.exists(filename):
        print(f"Baixando arquivo: {filename}...")
        try:
            urllib.request.urlretrieve(url, filename)
            print("Download concluído.")
            return True
        except Exception as e:
            print(f"ERRO ao baixar o arquivo {filename}: {e}")
            return False
    # else:
        # print(f"Arquivo {filename} já existe.") # Debug
    return True # Arquivo já existe


# --- Função de Detecção Facial e Desenho ---

def detect_and_draw_face(pil_image):
    """
    Detecta rostos em uma imagem PIL e desenha caixas ao redor deles.

    Args:
        pil_image (PIL.Image.Image): Imagem PIL de entrada.

    Retorna:
        tuple: (imagem PIL com caixas desenhadas, status da detecção: str).
               Retorna (imagem PIL original, status de erro) se houver falha.
    """
    # Converte PIL Image para OpenCV format (NumPy array BGR)
    try:
        # Convert to RGB, then to OpenCV BGR
        opencv_image = cv2.cvtColor(np.array(pil_image.convert('RGB')), cv2.COLOR_RGB2BGR)
        gray_image = cv2.cvtColor(opencv_image, cv2.COLOR_BGR2GRAY)
        gray_image = cv2.equalizeHist(gray_image) # Melhora o contraste

    except Exception as e:
        print(f"ERRO ao converter imagem para OpenCV/grayscale: {e}")
        return (pil_image, f"Erro processando imagem para detecção: {e}")


    # Tenta baixar o arquivo cascade se não existir
    if not download_cascade_file(HAAR_CASCADE_URL, HAAR_CASCADE_FACE_PATH):
        return (pil_image, f"Erro: Não foi possível baixar o arquivo de detecção facial.")

    # Carrega o classificador Haar Cascade
    face_cascade = cv2.CascadeClassifier(HAAR_CASCADE_FACE_PATH)

    if face_cascade.empty():
        print(f"ERRO: Classificador cascade vazio. O arquivo {HAAR_CASCADE_FACE_PATH} pode estar corrompido ou vazio.")
        return (pil_image, f"Erro: Classificador de detecção facial não carregado.")


    # Detecta rostos
    # scaleFactor: Reduz o tamanho da imagem em cada escala de imagem scanneada.
    # minNeighbors: Quantos vizinhos cada candidato a retângulo deve ter para reter o retângulo.
    # minSize: Tamanho mínimo possível do objeto. Objetos menores que isso são ignorados.
    faces = face_cascade.detectMultiScale(
        gray_image,
        scaleFactor=1.1, # Ajuste se necessário
        minNeighbors=5, # Ajuste se necessário
        minSize=(30, 30) # Ajuste se necessário
    )

    detection_status = f"Nenhum rosto detectado."

    # Desenha retângulos ao redor dos rostos detectados
    if len(faces) > 0:
        detection_status = f"{len(faces)} rosto(s) detectado(s)."
        color = (255, 0, 0) # Cor BGR para o retângulo (Azul)
        thickness = 2 # Espessura da linha

        for (x, y, w, h) in faces:
            cv2.rectangle(opencv_image, (x, y), (x+w, y+h), color, thickness)

    # Converte a imagem de volta para PIL format (RGB)
    try:
        processed_pil_image = PILImage.fromarray(cv2.cvtColor(opencv_image, cv2.COLOR_BGR2RGB))
        return (processed_pil_image, detection_status)
    except Exception as e:
        print(f"ERRO ao converter imagem OpenCV de volta para PIL: {e}")
        return (pil_image, f"{detection_status}. Erro na conversão final: {e}")


print("Variáveis globais, funções de persistência e a função detect_and_draw_face definidas.")

--- Célula 2: Imports Adicionais, Variáveis Globais e Função de Detecção Facial ---
Variáveis globais, funções de persistência e a função detect_and_draw_face definidas.


Célula 3: Funções de Configuração (API e Modelo)

Esta célula contém as definições das funções configurar_api_key e inicializar_modelo. A primeira busca a API Key dos segredos do Colab e configura o SDK do Google Generative AI. A segunda inicializa o modelo Gemini (gemini-1.5-flash-latest) e, crucialmente, inicia uma sessão de chat (chat_session). A sessão de chat é utilizada para manter o contexto da conversa (histórico real enviado à API) entre os turnos de interação na interface Gradio.

In [76]:
# Célula 3: Funções de Persistência

print("--- Célula 3: Funções de Persistência ---")

# --- Funções Auxiliares para Persistência ---

def load_history():
    """Carrega o log textual da conversa de um arquivo JSONL."""
    history = []
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, 'r') as f:
                for line in f:
                    if line.strip():
                        history.append(json.loads(line))
            # print(f"Log textual carregado de {HISTORY_FILE} para exibição.") # Debug
        except Exception as e:
            print(f"Erro ao carregar log textual de {HISTORY_FILE}: {e}")
            history = [] # Começa com histórico vazio em caso de erro
    # else:
        # print(f"Arquivo de log textual '{HISTORY_FILE}' não encontrado. Começando novo log.") # Debug
    return history

def save_history(history_entry):
    """Salva uma nova entrada no arquivo de log textual JSONL."""
    try:
        with open(HISTORY_FILE, 'a') as f: # 'a' para append
            json_line = json.dumps(history_entry)
            f.write(json_line + '\n')
    except Exception as e:
        print(f"Erro ao salvar log textual no arquivo {HISTORY_FILE}: {e}")

print("Funções de persistência (load/save_history) definidas.")

--- Célula 3: Funções de Persistência ---
Funções de persistência (load/save_history) definidas.


Célula 4: Funções de Persistência (Carregar/Salvar Log Textual)

Esta célula define as funções load_history e save_history. Estas funções implementam uma forma simples de persistência textual do log de interações, salvando cada turno (input do usuário e resposta do modelo) em um arquivo JSONL (burnoutcheck_history.jsonl) no ambiente do Colab. Este log é carregado ao iniciar a interface para exibir interações anteriores na mesma sessão do notebook.

In [77]:
# Célula 4: Funções de Configuração

print("--- Célula 4: Funções de Configuração ---")

def configurar_api_key():
    """Configura a API Key do Gemini usando segredos do Colab."""
    global api_key
    try:
        api_key = userdata.get('API_KEY')
        if not api_key:
             raise ValueError("API_KEY não encontrada nos segredos do Colab.")
        genai.configure(api_key=api_key)
        # print("API Key configurada com sucesso.") # Mensagem na próxima célula
        return True
    except ValueError as e:
        print(f"\nERRO na Configuração da API Key: {e}")
        print("Por favor, configure sua API Key nos segredos do Colab com o nome 'API_KEY'.")
        return False
    except Exception as e:
        print(f"\nOcorreu um erro inesperado ao configurar a API Key: {e}")
        return False


def inicializar_modelo():
    """Inicializa o modelo Gemini e a sessão de chat."""
    global model, chat_session, api_key
    if not api_key:
        # print("\nNão é possível inicializar o modelo sem uma API Key válida.") # Mensagem na próxima célula
        model = None
        chat_session = None
        return False

    try:
        model_name = 'gemini-1.5-flash-latest' # Modelo multimodal rápido
        model = genai.GenerativeModel(model_name)
        # Inicializa a sessão de chat AQUI para a sessão atual do Gradio.
        # O histórico real do Gemini começa agora.
        chat_session = model.start_chat(history=[])
        # print(f"Modelo Gemini '{model.model_name}' e sessão de chat inicializados.") # Mensagem na próxima célula
        return True
    except Exception as e:
        print(f"\nERRO ao inicializar o modelo Gemini '{model_name}' ou sessão de chat: {e}")
        print("Verifique se o nome do modelo está correto e se sua API Key está ativa e tem acesso a ele.")
        model = None
        chat_session = None
        return False

print("Funções de configuração (API e Modelo) definidas.")

--- Célula 4: Funções de Configuração ---
Funções de configuração (API e Modelo) definidas.


Célula 5: Funções de Análise e Disclaimer

Esta célula define funções auxiliares para o processamento principal:

  gerar_analise: Embora a chamada API tenha sido movida para a Célula 7, esta função (ou partes dela) pode ser mantida se outras lógicas de processamento de texto forem adicionadas aqui. Na estrutura final, a lógica principal de chamada API está em process_checkin.
  get_disclaimer_text: Gera o texto completo do aviso importante e ético em formato Markdown, com a opção de torná-lo enfático e incluir recursos de ajuda adicionais.
  check_for_severe_keywords: Uma heurística simples para verificar a presença de palavras-chave na resposta do modelo que possam indicar sofrimento, usada para disparar o aviso enfático.

In [78]:
# Célula 5 : Funções de Análise e Disclaimer

print("--- Célula 5: Funções de Análise e Disclaimer ---")

# --- Função de Geração de Análise ---

def gerar_analise(descricao, imagem, chat_session_obj):
    """
    Envia os inputs (texto + imagem) para o modelo Gemini usando ChatSession.
    Retorna: str (resposta do Gemini) ou None.
    """
    if not chat_session_obj:
        return None

    # --- Preparando e Enviando a Requisição Multimodal ---

    # Prompt Detalhado (Ajustado para que a resposta seja mais fácil de quebrar em seções)
    # Pedimos para ele USAR TÍTULOS ou marcadores claros para cada seção na resposta.
    prompt_instrucao = f"""
    Você é um assistente de check-in de bem-estar experimental. Analise o texto e a imagem fornecidos, considerando o contexto da nossa conversa (histórico incluído automaticamente pelo ChatSession).

    Seu foco é identificar *indícios gerais* de cansaço mental, estresse, tensão, baixa de humor, ansiedade ou irritabilidade a partir DOS INPUTS DESTE TURNO e do histórico. Sua análise é experimental e NÃO é um diagnóstico profissional.

    Em sua resposta para ESTE TURNO:
    * **SEJA CONCISO.** Vá direto ao ponto em cada seção.
    * **Formate sua resposta usando TÍTULOS ou marcadores claros para cada seção:**
        * Comece com: **"Avaliação Cautelosa:"** seguido pela sua análise.
        * Depois: **"Sugestões de Autocuidado:"** seguido por 2 a 3 sugestões GERAIS em *lista* (`* `).
        * Depois: **"Recursos Sugeridos:"** seguido por 1 ou 2 tipos de recursos/ajuda *com exemplos concretos* em *lista* (`* `).
    * **REFORCE:** Use linguagem **EXTREMAMENTE CAUTELOSA, NÃO-MÉDICA, NÃO-DIAGNÓSTICA**. Enfatize a natureza experimental.

    ---
    Descrição do usuário NESTE TURNO:
    {descricao}
    ---
    Análise Cautelosa e Sugestões para ESTE CHECK-IN (com contexto, se aplicável - NÃO É DIAGNÓSTICO, SEJA CONCISO, USE SEÇÕES MARCadas):
    """

    user_content_parts = [
        prompt_instrucao,
        imagem # Objeto PIL Image
    ]

    # Mensagem de processamento mostrada na função process_checkin do Gradio

    try:
        response = chat_session_obj.send_message(user_content_parts)

        if response and response.text:
            gemini_response_text = response.text
            # Não formatamos a quebra de linha aqui, Gradio Markdown lida melhor
            return gemini_response_text
        else:
            return "🚫 O modelo Gemini não gerou uma resposta de texto para esta requisição."

    except Exception as e:
        return f"❌ ERRO ao processar a requisição com o modelo Gemini: {e}"


# --- Função de Geração de Texto de Disclaimer (Retorna Markdown) ---

def get_disclaimer_text(enfatico=False):
    """
    Retorna o texto do aviso essencial e ético em formato Markdown.
    """
    disclaimer = ""
    if enfatico:
        disclaimer += "## 📢 ATENÇÃO ESPECIAL COM ESTE RESULTADO! 📢" # Markdown H2
        disclaimer += "\n\n**A análise detectou indícios que merecem sua ATENÇÃO.** **LEIA COM MUITA ATENÇÃO ABAIXO.**"

        # --- Recursos Fixos Relevantes (Incluídos com Alerta Enfático) ---
        disclaimer += "\n\n### RECURSOS IMPORTANTES DE AJUDA (BUSQUE UM PROFISSIONAL!)" # Markdown H3
        disclaimer += "\n* **CVV (Centro de Valorização da Vida):** Ligue 188 (gratuito e sigiloso em todo Brasil)"
        disclaimer += "\n* **Emergência (SAMU):** Ligue 192 em casos de crise aguda (risco a si ou a outros)"
        disclaimer += "\n* **Centros de Atenção Psicopsocial (CAPS):** Serviços de saúde mental pública (busque online pelo CAPS mais próximo na sua cidade)"
        disclaimer += "\n* **Pronto-socorro Psiquiátrico ou Hospital Geral:** Para emergências em saúde mental grave."
        disclaimer += "\n* **Psicólogos e Psiquiatras Particulares:** Consulte planos de saúde, clínicas ou convênios."

    else:
         disclaimer += "## AVISO IMPORTANTE E ÉTICA" # Markdown H2

    disclaimer += "\n\nEste é um projeto EXPERIMENTAL e educacional, criado para demonstração de IA."
    disclaimer += "\nA análise fornecida acima é gerada por um modelo de IA e se baseia em padrões aprendidos, NÃO em conhecimento médico/psicológico profissional."
    disclaimer += "\n\n### 🚫🚫🚫 ESTA FERRAMENTA NÃO FAZ DIAGNÓSTICOS! 🚫🚫🚫" # Markdown H3 com emojis
    disclaimer += "\n\nSe você está genuinamente preocupado com sua saúde mental ou física, se sente sintomas de burnout, estresse crônico ou qualquer outro problema, **PROCURE SEMPRE um PROFISSIONAL DE SAÚDE QUALIFICADO (médico, psicólogo, psiquiatra).**"
    disclaimer += "\n\nUse esta ferramenta APENAS para fins demonstrativos e de check-in pessoal *com extrema cautela e consciência de suas limitações*."

    return disclaimer

# --- Lógica para Alerta Cauteloso Extra Baseado em Palavras-Chave ---
# (Mesma lógica da Célula 5 anterior)

sofrimento_keywords = [
    "sinais significativos", "muito cansado", "profundamente",
    "ansiedade alta", "bastante estresse", "dificuldade enorme",
    "sobrecarregado", "insuportável", "exaustão", "desesperança",
    "crise", "emergência", "piora", "agrava", "descontrolado", "pânico",
    "não aguento", "desistir", "ferimento", "dor insuportável", "suicida", "morte", "machucar" # Adicionadas mais palavras fortes
]

def check_for_severe_keywords(text):
    """Verifica se o texto contém palavras-chave indicando sofrimento."""
    if not text:
        return False
    return any(keyword in text.lower() for keyword in sofrimento_keywords)


print("Funções de análise e disclaimer definidas (adaptadas para Markdown).")

--- Célula 5: Funções de Análise e Disclaimer ---
Funções de análise e disclaimer definidas (adaptadas para Markdown).


Célula 6: Execução da Configuração Inicial

Esta célula executa as funções de configuração (configurar_api_key, inicializar_modelo) definidas nas células anteriores e carrega o log de histórico textual (load_history). A saída desta célula mostrará o status dessas etapas cruciais. É essencial que esta célula execute com sucesso antes de tentar lançar a interface.

In [79]:
# Célula 6: Executar Configuração Inicial

print("--- Célula 6: Executando Configuração e Inicialização ---")

# Configurar API Key
config_api_ok = configurar_api_key()

if config_api_ok:
    print("Configuração da API Key OK.")

    # Inicializar Modelo e Sessão de Chat Gemini
    config_model_ok = inicializar_modelo() # Inicializa modelo E chat_session

    if config_model_ok:
        print(f"Modelo Gemini '{model.model_name}' e sessão de chat inicializados com sucesso.")

        # Carregar Log Textual de Histórico
        chat_history_list = load_history() # Carrega o log salvo anteriormente
        print(f"Log textual carregado de {len(chat_history_list)//2} check-in(s) anterior(es).")

        print("Execução da Célula 6 concluída. Configuração OK.")

    else:
        print("ERRO na inicialização do modelo Gemini.")
else:
    print("ERRO na configuração da API Key.")

# As variáveis globais (api_key, model, chat_session, chat_history_list) estão agora configuradas.

--- Célula 6: Executando Configuração e Inicialização ---
Configuração da API Key OK.
Modelo Gemini 'models/gemini-1.5-flash-latest' e sessão de chat inicializados com sucesso.
Log textual carregado de 14 check-in(s) anterior(es).
Execução da Célula 6 concluída. Configuração OK.


Célula 7: Função Principal de Processamento para o Gradio

Esta é uma das células mais importantes em termos de lógica. A função process_checkin atua como o controlador da aplicação. Ela é vinculada ao botão "Enviar Check-in" na interface Gradio.

Ao ser chamada:

  Recebe a descrição textual e o caminho do arquivo da imagem do Gradio.
   Realiza validações básicas.
   Chama a função detect_and_draw_face (da Célula 2) para processar a imagem, obter o status da detecção facial e a imagem com as marcações.
  Constrói o prompt completo para o modelo Gemini, incluindo o status da detecção facial como informação preliminar. O prompt é cuidadosamente elaborado para solicitar que o Gemini formate sua Avaliação Cautelosa como uma lista de pontos Markdown (*) e formate as sugestões e recursos também como listas.
  Chama a API do Gemini (chat_session.send_message) com o prompt e a imagem original para análise multimodal.
  Processa a resposta textual completa do Gemini, dividindo-a nas seções esperadas (Análise Cautelosa, Sugestões de Autocuidado, Recursos Sugeridos) com base em marcadores no texto.
  Aplica um pós-processamento de limpeza (clean_text_part) para remover artefatos indesejados como ** e literais \n\n das partes extraídas.
  Verifica palavras-chave na resposta completa para determinar se o aviso ético enfático deve ser exibido.
  Prepara o texto para cada componente de saída no formato Markdown (adicionando títulos H3).
   Salva o log textual da interação atual no arquivo (.jsonl).
  Retorna uma tupla de 6 valores contendo o conteúdo formatado (Markdown ou Imagem) para cada um dos componentes de saída na interface Gradio.

In [80]:
# Célula 7 : Função Principal de Processamento

print("--- Célula 7: Função Principal de Processamento (Gradio - Análise em Lista) ---")

def process_checkin(sentimentos_text, selfie_image_path):
    """
    Processa o check-in multimodal (texto + imagem), executa detecção facial,
    chama o Gemini com prompt refinado para análise em lista.
    Retorna uma tupla de valores para os componentes de saída.
    """
    global chat_history_list, chat_session, model

    analysis_md = "Processando..."
    tips_md = "Aguardando sugestões..."
    resources_md = "Aguardando recursos..."
    disclaimer_md = ""
    log_output_for_box = ""
    processed_image_output = None

    # --- Validação Inicial ---
    if not model or not chat_session:
         error_msg = "Erro: Modelo Gemini ou sessão de chat não inicializados. Execute as células de configuração antes de usar a interface."
         disclaimer_md = get_disclaimer_text(enfatico=False)
         return (f"Erro de Configuração: {error_msg}", "", "", disclaimer_md, processed_image_output, "")

    descricao_sentimentos = sentimentos_text.strip()
    if not descricao_sentimentos:
        error_msg = "Por favor, descreva como você se sente na caixa de texto antes de enviar."
        disclaimer_md = get_disclaimer_text(enfatico=False)
        return (f"Erro de Entrada: {error_msg}", "", "", disclaimer_md, processed_image_output, "")

    if descricao_sentimentos.lower() == 'sair':
        return ("Sessão encerrada a pedido do usuário. Cuide-se!", "", "", "", processed_image_output, "")

    # --- Processar Imagem (Abrir e Detectar Face) ---
    pil_image_original = None
    if selfie_image_path is None:
         error_msg = "Por favor, faça o upload de uma selfie para análise."
         disclaimer_md = get_disclaimer_text(enfatico=False)
         return (f"Erro de Entrada: {error_msg}", "", "", disclaimer_md, processed_image_output, "")

    try:
        pil_image_original = PILImage.open(selfie_image_path)
    except Exception as e:
        error_msg = f"Erro ao abrir a imagem do Gradio: Verifique o arquivo. Detalhe: {e}"
        disclaimer_md = get_disclaimer_text(enfatico=False)
        return (f"Erro de Imagem: {error_msg}", "", "", disclaimer_md, processed_image_output, "")

    # --- Executar Detecção Facial ---
    try:
        processed_pil_image, detection_status = detect_and_draw_face(pil_image_original)
        processed_image_output = processed_pil_image

    except Exception as e:
        error_msg = f"Erro inesperado durante a detecção facial: {e}"
        disclaimer_md = get_disclaimer_text(enfatico=False)
        return (f"Erro de Detecção Facial: {error_msg}", "", "", disclaimer_md, pil_image_original, "")


    # --- Mensagem de Processamento ---
    analysis_md = f"Status Detecção Facial: {detection_status}\n🧠 Enviando para Análise do Gemini... Aguarde. 🧠"

    # --- Gerar Análise com o Modelo Gemini (Prompt para Lista) ---
    full_prompt_with_status = f"""
    Você é um assistente de check-in de bem-estar experimental. Analise o texto, a informação de detecção visual preliminar E a imagem fornecidos, considerando o contexto da nossa conversa (histórico incluído automaticamente pelo ChatSession).

    Sua análise é experimental e NÃO é um diagnóstico profissional.

    Considere as seguintes informações para ESTE TURNO:
    1. A descrição textual do usuário: {descricao_sentimentos}
    2. Uma detecção visual preliminar automatizada: {detection_status}
    3. A imagem fornecida (para análise visual holística que vai além da detecção básica).
    4. O histórico da conversa até agora.

    Em sua resposta para ESTE TURNO:
    * **SEJA CONCISO.** Vá direto ao ponto.
    * **Formate sua resposta usando TÍTULOS claros e Markdown para CADA SEÇÃO.**
        * Comece com: **"Avaliação Cautelosa:"**
          * **Formate a análise CAUTELOSA como uma lista concisa de 3 a 5 observações principais usando marcadores de lista Markdown (`* `)**. A **clareza e a organização em pontos** são a prioridade nesta seção. Cada ponto deve ser uma observação distinta.
        * Depois: **"Sugestões de Autocuidado:"** seguido por 2 a 3 sugestões GERAIS em *lista* (`* `).
        * Depois: **"Recursos Sugeridos:"** seguido por 1 ou 2 tipos de recursos/ajuda *com exemplos concretos* em *lista* (`* `).
    * **REFORCE:** Use linguagem **EXTREMAMENTE CAUTELOSA, NÃO-MÉDICA, NÃO-DIAGNÓSTICA**. Enfatize a natureza experimental.
    """

    user_content_parts_with_status = [
        full_prompt_with_status,
        pil_image_original
    ]

    gemini_full_response_text = None

    try:
        response = chat_session.send_message(user_content_parts_with_status)
        if response and response.text:
            gemini_full_response_text = response.text
        else:
            gemini_full_response_text = "🚫 O modelo Gemini não gerou uma resposta de texto para esta requisição."
    except Exception as e:
        gemini_full_response_text = f"❌ ERRO ao processar a requisição com o modelo Gemini: {e}"


    # --- Processar Resposta Completa do Gemini e Dividir em Seções (Pós-processamento de Limpeza e Formatação) ---
    analysis_part = ""
    tips_part = ""
    resources_part = ""
    final_disclaimer_md = ""
    log_output_for_box = ""


    if gemini_full_response_text and not gemini_full_response_text.startswith(("🚫", "❌")):
        # Tenta encontrar as seções e separar
        parts = gemini_full_response_text.split('Sugestões de Autocuidado:')
        analysis_part_raw = parts[0]
        # Extrai a parte da análise após o título "Avaliação Cautelosa:"
        # Remove o título e pega o resto
        analysis_part = analysis_part_raw.split("Avaliação Cautelosa:", 1)[-1].strip() if "Avaliação Cautelosa:" in analysis_part_raw else analysis_part_raw.strip()


        if len(parts) > 1:
            tips_resources_part = parts[1].split('Recursos Sugeridos:')
            tips_part = tips_resources_part[0].strip() if tips_resources_part else ""
            resources_part = tips_resources_part[1].strip() if len(tips_resources_part) > 1 else ""
        else:
             tips_part = ""
             resources_part = ""

        # --- PÓS-PROCESSAMENTO DE LIMPEZA (Remove artefatos) ---
        # A função de limpeza agora apenas remove artefatos e não insere parágrafos/listas heuristicamente
        def clean_text_part(text):
             cleaned_lines = []
             # Substitui \n\n literal por quebra de linha real (se ainda aparecer)
             text = text.replace('\\n\\n', '\n\n')
             # Divide em linhas para limpar artefatos
             for line in text.splitlines():
                 stripped_line = line.strip()
                 # Remove linhas que são apenas ** ou vazias ou outros artefatos comuns
                 if stripped_line == "" or stripped_line in ["**", "_", "*", "#", "-", "=", "+"]:
                     continue
                 cleaned_lines.append(stripped_line)

             # Junta as linhas limpas de volta com quebras de linha normais.
             # A formatação de lista ou parágrafo virá do Gemini (se ele seguir o prompt).
             return "\n".join(cleaned_lines).strip()


        # Aplica a limpeza simples em todas as partes
        analysis_part_cleaned = clean_text_part(analysis_part)
        tips_part_cleaned = clean_text_part(tips_part)
        resources_part_cleaned = clean_text_part(resources_part)


        # Formata as partes limpas como Markdown (Adiciona títulos H3)
        # A formatação INTERNA (a lista * ) deve vir do Gemini.
        final_analysis_md = "### Análise Cautelosa\n" + analysis_part_cleaned
        final_tips_md = "### Sugestões de Autocuidado\n" + tips_part_cleaned # A lista * deve vir do Gemini
        final_resources_md = "### Recursos Sugeridos\n" + resources_part_cleaned # A lista * deve vir do Gemini


        # --- Lógica para Alerta Cauteloso Extra ---
        is_severe = check_for_severe_keywords(gemini_full_response_text)
        final_disclaimer_md = get_disclaimer_text(enfatico=is_severe)

        # --- Preparar para Salvar Histórico (Simples) ---
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_user_text_summary = f"Você: {descricao_sentimentos} ({detection_status}) [Imagem Enviada]"
        # Usa o texto LIMPO da análise para o resumo no log
        log_model_text_summary = f"Gemini IA: {analysis_part_cleaned.splitlines()[0]}..." if analysis_part_cleaned else "Gemini IA: (Sem análise gerada)..."


        user_log_entry = {"timestamp": timestamp, "role": "user", "text": log_user_text_summary, "image_uploaded": True}
        model_log_entry = {"timestamp": timestamp, "role": "model", "text": gemini_full_response_text} # Salva a resposta COMPLETA no log
        save_history(user_log_entry)
        save_history(model_log_entry)

        # Log para exibir na UI (Markdown formatado)
        log_output_for_box = f"**[{timestamp}] Você:** {log_user_text_summary}\n"
        log_output_for_box += f"**[{timestamp}] Gemini IA:** {log_model_text_summary}\n\n"


        # Retorna os outputs formatados E a imagem processada E o log deste turno
        return (final_analysis_md, final_tips_md, final_resources_md, final_disclaimer_md, processed_image_output, log_output_for_box)


    else: # Caso a resposta do Gemini seja um erro ou None
         error_analysis_md = gemini_full_response_text
         final_disclaimer_md = get_disclaimer_text(enfatico=False)

         # Prepara log para erro
         timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
         log_user_text_summary = f"Você: {descricao_sentimentos} ({detection_status}) [Imagem Enviada]"
         log_output_for_box = f"**[{timestamp}] Você:** {log_user_text_summary}\n"
         log_output_for_box += f"**[{timestamp}] Erro no turno:** {error_analysis_md}\n\n"

         return (error_analysis_md, "", "", final_disclaimer_md, processed_image_output, log_output_for_box)


print("Função principal de processamento (process_checkin) atualizada (prompt para lista na análise).")

--- Célula 7: Função Principal de Processamento (Gradio - Análise em Lista) ---
Função principal de processamento (process_checkin) atualizada (prompt para lista na análise).


Célula 8: Criação e Lançamento da Interface Gradio

Esta é a célula final que define e lança a interface web interativa usando Gradio. Utiliza gr.Blocks para criar um layout estruturado com linhas (gr.Row) e colunas (gr.Column).

  Define componentes de input (gr.Textbox, gr.Image) e de output (gr.Markdown para texto formatado, gr.Image para a imagem processada, gr.Textbox para o log).
  Usa CSS (css parameter in gr.Blocks) para centralizar o título principal (h1).
   Define as dimensões (altura, linhas) dos componentes onde possível e consistente com a versão do Gradio utilizada.
   Vincula os botões (send_button, exit_button) à função process_checkin e a outras funções usando o método .click(), especificando quais componentes fornecem inputs e quais componentes serão atualizados (outputs) com os valores retornados por process_checkin.
   Inicia a interface chamando demo.launch(share=True, ...), o que gera o link público (.gradio.live) para acessar a ferramenta em uma nova aba do navegador. A célula permanecerá em execução enquanto a interface estiver ativa.

In [81]:
# Célula 8 : Criação e Lançamento da Interface Gradio

print("--- Célula 8: Criação e Lançamento da Interface Gradio ---")

# Verifica se a configuração e inicialização foram bem-sucedidas nas células anteriores
if model is not None and chat_session is not None:
    print("Configuração OK. Criando interface Gradio...")

    # Carregar e formatar histórico textual inicial para exibir na interface Gradio
    initial_history_log_text = "" # String simples para o valor inicial do Textbox
    loaded_log = load_history() # Carrega o log do arquivo
    if loaded_log:
        initial_history_log_text += "="*60 + "\n"
        initial_history_log_text += "### Histórico Textual de Check-ins Anteriores (Log) ###\n"
        initial_history_log_text += "="*60 + "\n\n"
        for entry in loaded_log:
            role = "Você" if entry['role'] == 'user' else ("Gemini IA" if entry['role'] == 'model' else "Erro")
            img_status = " [Imagem Enviada]" if entry.get("image_uploaded") else ""
            text_content = entry['text']
            wrapped_text = "\n".join(textwrap.wrap(text_content, width=80)) # Quebra linhas

            initial_history_log_text += f"**[{entry['timestamp']}] {role}:**{img_status}\n{wrapped_text}\n\n" # Formato Markdown para o log inicial
        initial_history_log_text += "="*60 + "\n" # Separador após o histórico


    initial_history_log_text += "--- Fim Histórico Carregado ---\n\n"
    initial_history_log_text += "Log desta sessão abaixo.\n\n" # Mensagem inicial


    # --- Criação da Interface Gradio Usando Blocks Diretamente ---

    # Adicionar CSS para centralizar o título H1 e maybe ajustar espaçamento/bordas se necessário
    css_theme = """
    h1 { text-align: center; } /* Centraliza o título H1 */
    /* Adicione mais CSS aqui para refinar a aparência se desejar */
    /* Ex: adicionar bordas, ajustar padding/margin das caixas */
    /* .gradio-container { border: 1px solid #ccc; padding: 20px; } */
    """

    with gr.Blocks(title="BurnoutCheck IA", css=css_theme) as demo: # Adiciona o CSS aqui
        gr.Markdown("# BurnoutCheck IA - Check-in Multimodal")
        gr.Markdown("Ferramenta experimental de check-in de bem-estar usando IA multimodal (texto + imagem) do Gemini. **NÃO é um diagnóstico médico.**")

        # Componentes de Entrada
        textbox_lines = 6 # Ajuste este número para chegar perto da altura da imagem
        image_height = 200 # Altura desejada para a imagem

        sentimentos_input = gr.Textbox(
            label="1. Descreva seus sentimentos:",
            lines=textbox_lines, # Usando 'lines' para altura do Textbox
            placeholder='Descreva como você se sente (cansaço, estresse, etc.). Digite "sair" para encerrar sessão interativa.',
            interactive=True,
            # height=... # Removido pois causa erro em algumas versões
        )

        # Mantendo 'height' para Image (se der erro, remover)
        selfie_input = gr.Image(
            label="2. Envie sua Selfie:",
            type="filepath", # Gradio fornece caminho do arquivo
            interactive=True,
            height=image_height, # Mantendo height para Image
            # source="webcam" # Opcional: usar webcam diretamente no navegador Gradio
        )


        # Botões
        send_button = gr.Button(value='Enviar Check-in', interactive=True)
        exit_button = gr.Button(value='Sair', interactive=True)


        # --- COMPONENTES DE SAÍDA ---
        analysis_box = gr.Markdown(label="Resultados da Análise Cautelosa")
        tips_box = gr.Markdown(label="Sugestões de Autocuidado")
        resources_box = gr.Markdown(label="Recursos Sugeridos")
        disclaimer_box = gr.Markdown(label="Aviso Importante e Ética")
        processed_image_box = gr.Image(label="Selfie Processada com Detecção Facial", interactive=False, height=image_height) # Altura igual à imagem de input

        # O log histórico agora é um Chatbot component, que é mais visualmente agradável para histórico de conversas
        # Ou podemos continuar com Textbox e append. Vamos manter Textbox e append de strings formatadas.
        history_log_box = gr.Textbox(
            label="Log de Interações Desta Sessão",
            lines=15, # Aumenta as linhas para exibir mais log
            interactive=False,
            value=initial_history_log_text, # Valor inicial com log carregado
            autoscroll=True,
            # render_as="html" # Opcional: se quiser HTML mais complexo no log
        )


        # --- Layout da Interface (Organizando Componentes) ---

        with gr.Row(): # Inputs na mesma linha
            with gr.Column(): # Coluna para texto
                 sentimentos_input
            with gr.Column(): # Coluna para imagem
                 selfie_input

        with gr.Row(): # Botões na mesma linha
             send_button
             exit_button

        gr.Markdown("---") # Adiciona uma linha horizontal

        # Seções de saída organizadas em colunas ou verticalmente
        with gr.Column():
            # Imagem processada ao lado da análise
            with gr.Row():
                 with gr.Column(scale=2):
                      analysis_box # Análise textual aqui
                 with gr.Column(scale=1):
                      processed_image_box # Imagem processada aqui

            with gr.Row(): # Dicas e Recursos lado a lado
                 with gr.Column():
                      tips_box # Dicas aqui
                 with gr.Column():
                      resources_box # Recursos aqui

            disclaimer_box # Disclaimer aqui

            gr.Markdown("---") # Separador antes do log

            history_log_box # Log aqui


        # --- Vínculo entre Evento (Clique no Botão) e Função de Processamento ---
        # O log_output_for_box agora é uma STRING para ser APENDADA ao valor existente do history_log_box
        send_button.click(
            fn=process_checkin,
            inputs=[sentimentos_input, selfie_input],
            # Note que history_log_box está na lista de outputs, process_checkin retorna a string para adicionar
            outputs=[analysis_box, tips_box, resources_box, disclaimer_box, processed_image_box, history_log_box],
            # Para atualizar o Textbox adicionando texto, usamos o estado.
            # O Gradio precisa saber o valor ATUAL do history_log_box para adicionar a nova string.
            # Isso requer um input adicional invisível ou o uso do parâmetro state.
            # Forma mais simples: retornar a string para o Textbox e configurar o Textbox para append.
            # O Textbox do Gradio, por padrão, substitui o conteúdo. Para append, precisaria de JS ou State.
            # Vamos ajustar process_checkin para retornar o *conteúdo total* do log box (antigo + novo)
            # ou usar state. Usar state é mais "senior dev".

            # Usando State:
            # Requer que process_checkin receba o estado atual do log box como input
            # e retorne o novo estado (string completa atualizada) para o log box output.
            # Vamos adaptar process_checkin.
            # inputs=[sentimentos_input, selfie_input, history_log_box], # Adiciona o log atual como input
            # outputs=[analysis_box, tips_box, resources_box, disclaimer_box, processed_image_box, history_log_box] # Log é input E output

            # ALTERNATIVA MAIS SIMPLES E COMUM: Ter uma função SEPARADA só para o log.
            # Mas o log depende do resultado da análise... vamos tentar retornar a string completa e torcer pelo autoscroll.
            # Se não funcionar, a versão "senior" usa State.

            # Voltando à abordagem de retornar a string completa (agora formatada) para o log box.
            # O Textbox com autoscroll deve rolar para o final.

        )

        # Vínculo do botão Sair
        exit_button.click(
             fn=lambda: gr.update(interactive=False),
             inputs=[],
             outputs=[exit_button]
        )


    # --- Lançamento da Interface (Usando o objeto 'demo' de Blocks) ---
    print("Lançando interface Gradio (com Blocks refatorado e Detecção Facial)... Clique no link público que aparecerá abaixo.")
    launch_info = demo.launch(share=True, inline=False, debug=True)

    print("\nInterface Gradio lançada.")

--- Célula 8: Criação e Lançamento da Interface Gradio ---
Configuração OK. Criando interface Gradio...
Lançando interface Gradio (com Blocks refatorado e Detecção Facial)... Clique no link público que aparecerá abaixo.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0f4c3b773d4adcbd51.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0f4c3b773d4adcbd51.gradio.live

Interface Gradio lançada.


Guia de Utilização da Interface Gradio

Após executar a Célula 8, um link público (https://...gradio.live) aparecerá na saída da célula. Clique neste link para abrir a interface web em uma nova aba do seu navegador.

A interface é dividida nas seguintes áreas:

    1. Descreva seus sentimentos: Caixa de texto para você digitar como se sente (cansaço, estresse, humor, etc.).
    2. Envie sua Selfie: Área para fazer upload da sua selfie. Clique para selecionar um arquivo ou arraste e solte a imagem.
    3. Enviar Check-in: Botão para processar seus inputs. Ao clicar, a imagem será processada (detecção facial), e o texto + imagem serão enviados para a análise do Gemini. A interface mostrará um indicador de processamento.
    4. Sair: Botão para desativar a interface (apenas desabilita o botão neste protótipo). Para parar completamente a interface e o servidor Gradio, interrompa a execução da Célula 8 no Colab (clicando no quadrado cinza ao lado dela).
    5. Resultados da Análise Cautelosa: Exibirá a análise textual gerada pelo Gemini baseada nos seus inputs e no histórico. Espera-se que seja uma lista de pontos para facilitar a leitura.
    6. Sugestões de Autocuidado: Exibirá as sugestões gerais de autocuidado fornecidas pelo Gemini em formato de lista.
    7. Recursos Sugeridos: Exibirá sugestões de tipos de recursos ou ajuda, com exemplos concretos, fornecidas pelo Gemini em formato de lista.
    8. Aviso Importante e Ética: Exibirá o disclaimer essencial. Será exibido de forma mais enfática (com recursos diretos de ajuda) se a análise do Gemini detectar indícios que a heurística de palavras-chave considerar de atenção.
    9. Selfie Processada com Detecção Facial: Exibirá a imagem da selfie que você enviou com um retângulo desenhado ao redor do(s) rosto(s) detectado(s pela etapa de Visão Computacional).
    10. Log de Interações Desta Sessão: Exibirá um resumo das interações (seu input e a primeira linha da resposta do Gemini) para esta sessão do notebook.

Notas Técnicas e Desafios Superados

Este projeto, embora um protótipo, envolveu a integração de diferentes tecnologias e a superação de desafios comuns no desenvolvimento com LLMs e interfaces rápidas:

    - Integração Multimodal: Combinar texto e imagem como input para o modelo Gemini.
    - Visão Computacional Integrada: Adicionar uma etapa de processamento visual (detecção facial com OpenCV) antes da análise do LLM, enriquecendo o processo e fornecendo feedback visual explícito (imagem processada).
    - Construção de UI Interativa em Notebook: Utilizar Gradio para criar uma interface web funcional diretamente de um ambiente de notebook, com layout estruturado.
    - Versionamento de Bibliotecas: Lidar com incompatibilidades de API entre diferentes versões do Gradio (loading_indicator, button_style, métodos de layout como gr.Layout vs. parâmetros diretos) e adaptar o código para a versão disponível no ambiente Colab.
    - Formatação de Output de LLM: O desafio de fazer o modelo Gemini gerar texto com formatação interna específica (parágrafos, listas) de forma consistente, e a implementação de pós-processamento para limpar artefatos (**, \n\n literal) e garantir alguma estrutura (Análise em lista) quando o prompt não é seguido perfeitamente.
    - Gerenciamento de Estado (Simples): Uso de chat_session para histórico da API e salvamento/carregamento simples em arquivo (.jsonl) para persistência básica do log textual entre sessões do notebook.

Melhorias Futuras Potenciais

Este protótipo pode ser expandido de diversas maneiras:

    * Análise Visual Mais Avançada:
        - Detecção de pontos faciais (landmarks).
        - Tentativa de análise de expressão facial mais granular (felicidade, tristeza, etc. - com as devidas ressalvas e avisos éticos severos).
        - Análise do fundo da imagem (ambiente).
    * Integração de Dados Adicionais:
        - Inputs sobre qualidade do sono, nível de - energia, hábitos alimentares/exercícios.
        - Integração com APIs de clima, horário local.
    * Histórico de Chat Real: Implementar um histórico de chat mais completo e visualmente agradável na interface (usando gr.Chatbot) e gerenciar o histórico da chat_session de forma mais robusta.
    * Banco de Dados: Persistir o histórico e dados do usuário em um banco de dados real (SQL, NoSQL).
    * Sugestões Dinâmicas: Usar a análise do Gemini para gerar sugestões de autocuidado e recursos mais personalizadas com base no que foi identificado.
    * Recursos Interativos: Links clicáveis para recursos sugeridos.
    * Feedback Visual Aprimorado: Indicador de carregamento mais proeminente, desabilitar inputs/botão durante o processamento.
    * Tematização: Aplicar um tema Gradio customizado para um visual mais distinto.
    * Implantação: Implantar a aplicação em um serviço de hospedagem contínuo (como Hugging Face Spaces, Render, etc.) para que não dependa da sessão Colab.